# Web Research Agent

Build an agent that browses real websites, extracts structured data, and returns validated Pydantic objects.

Uses [Cloudflare Browser Run](https://developers.cloudflare.com/browser-run/) — serverless headless Chrome. No Selenium, no local browser.

**Prerequisites:**
- Cloudflare API token with **Workers AI → Read** and **Browser Rendering → Edit** permissions
- See [01_getting_started.ipynb](01_getting_started.ipynb) for setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

assert os.getenv("CLOUDFLARE_ACCOUNT_ID"), "Set CLOUDFLARE_ACCOUNT_ID"
assert os.getenv("CLOUDFLARE_API_TOKEN"), "Set CLOUDFLARE_API_TOKEN"

## Browse any webpage as markdown

The `browse` tool fetches a URL and returns clean markdown, even for JS-rendered SPAs.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai_cloudflare import BrowserRunToolset

agent = Agent(
    "cloudflare:@cf/meta/llama-3.3-70b-instruct-fp8-fast",
    toolsets=[BrowserRunToolset(tools=["browse"])],
    system_prompt="Summarize web pages concisely when asked.",
)

result = agent.run_sync(
    "Go to https://developers.cloudflare.com/browser-run/ and tell me "
    "what Browser Run is in 2-3 sentences."
)
print(result.output)

## Extract structured data from a real website

The `extract` tool uses AI to pull structured JSON from any rendered page. One API call.

In [ ]:
from pydantic import BaseModel

class PricingPlan(BaseModel):
    name: str
    price: str
    features: list[str]

class PricingPage(BaseModel):
    company: str
    plans: list[PricingPlan]
    has_free_tier: bool

agent = Agent(
    "cloudflare:@cf/meta/llama-3.3-70b-instruct-fp8-fast",
    output_type=PricingPage,
    toolsets=[BrowserRunToolset(tools=["browse", "extract"])],
    system_prompt="Extract pricing information from web pages.",
)

result = agent.run_sync(
    "Analyze pricing from https://www.cloudflare.com/plans/"
)

pricing = result.output
print(f"Company: {pricing.company}")
print(f"Free tier: {pricing.has_free_tier}")
for plan in pricing.plans:
    print(f"\n{plan.name}: {plan.price}")
    for f in plan.features[:3]:
        print(f"  - {f}")

## Discover and follow links

The `discover_links` tool finds all links on a page. Useful for research agents that need to explore.

In [ ]:
# Use the toolset directly (not through an agent) for quick exploration
toolset = BrowserRunToolset(tools=["discover_links", "browse"])

# Find links on the Browser Run docs
links = await toolset._discover_links(
    "https://developers.cloudflare.com/browser-run/"
)
print(f"Found {len(links.split(chr(10)))} links")

# Filter to quick-actions pages
qa_links = [l for l in links.split("\n") if "/quick-actions/" in l and l.endswith("/")]
print(f"\nQuick Actions pages:")
for link in qa_links[:5]:
    print(f"  {link}")

## Scrape specific elements

When you only need specific parts of a page, use CSS selectors.

In [ ]:
toolset = BrowserRunToolset()

result = await toolset._scrape(
    "https://www.cloudflare.com",
    selectors=["h1", "h2"],
)
print(result)

## All Browser Run tools

| Tool | What it does | Use case |
|------|-------------|----------|
| `browse` | Fetch page as markdown | Reading any web page |
| `extract` | AI-powered JSON extraction | Pulling structured data |
| `crawl` | Crawl entire sites | Building knowledge bases |
| `scrape` | CSS selector extraction | Grabbing specific elements |
| `discover_links` | Find all links | Research exploration |
| `screenshot` | Page screenshot (PNG) | Visual QA, multimodal |

All powered by [Browser Run Quick Actions](https://developers.cloudflare.com/browser-run/quick-actions/) — no local browser, no Selenium, no Playwright setup.